<a href="https://colab.research.google.com/github/AmnaNoor123/urdu-ocr-codesaviours-si26-amna/blob/main/SI26_Week3_Amna.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/AmnaNoor123/urdu-ocr-codesaviours-si26-amna.git
!cp urdu-ocr-codesaviours-si26-amna/labels.csv data/labels.csv

fatal: destination path 'urdu-ocr-codesaviours-si26-amna' already exists and is not an empty directory.


In [2]:
!find urdu-ocr-codesaviours-si26-amna -maxdepth 2 -type f

urdu-ocr-codesaviours-si26-amna/SI26_Week2_amna.ipynb
urdu-ocr-codesaviours-si26-amna/.git/HEAD
urdu-ocr-codesaviours-si26-amna/.git/description
urdu-ocr-codesaviours-si26-amna/.git/config
urdu-ocr-codesaviours-si26-amna/.git/index
urdu-ocr-codesaviours-si26-amna/.git/packed-refs
urdu-ocr-codesaviours-si26-amna/labels.csv
urdu-ocr-codesaviours-si26-amna/README.md
urdu-ocr-codesaviours-si26-amna/SI26_Week1_Amna.ipynb


In [3]:
import os, shutil
os.makedirs('data', exist_ok=True)
shutil.copy('urdu-ocr-codesaviours-si26-amna/labels.csv', 'data/labels.csv')

import pandas as pd
df = pd.read_csv('data/labels.csv')
print('Total entries:', len(df))

Total entries: 200


In [4]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile
with zipfile.ZipFile('/content/drive/MyDrive/Books.zip', 'r') as zip_ref:
    zip_ref.extractall('data/raw')
import os
print('Unzipped folders:', os.listdir('data/raw'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Unzipped folders: ['Newspaper', 'Books', 'Handwritten', 'Other', 'Sign boards', 'Synthetic raw images']


In [5]:
import re
def clean_filename(name):
    base, ext = os.path.splitext(name)
    def decode_match(m):
        return chr(int(m.group(1), 16))
    base = re.sub(r'#U([0-9a-fA-F]{4,6})', decode_match, base)
    ext = re.sub(r'#U([0-9a-fA-F]{4,6})', decode_match, ext)
    base = re.sub(r'\s*\(\d+\)$', '', base)
    full = base + ext
    full = re.sub(r'(\.\w+)\1$', r'\1', full)
    return full

for root, dirs, files in os.walk('data/raw'):
    for fname in files:
        new_name = clean_filename(fname)
        if new_name != fname:
            os.rename(os.path.join(root, fname), os.path.join(root, new_name))
print('Filenames cleaned')

Filenames cleaned


Dataset Class

In [6]:
!pip install transformers torch pillow pandas

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor
from PIL import Image

class UrduOCRDataset(Dataset):
    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f'Dataset loaded: {len(self.data)} samples')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row['image']).convert('RGB')
        encoding = self.processor(image, return_tensors='pt')
        pixel_values = encoding.pixel_values.squeeze()

        labels = self.processor.tokenizer(
            row['text'],
            padding='max_length',
            max_length=128
        ).input_ids
        labels = torch.tensor(labels)

        return {'pixel_values': pixel_values, 'labels': labels}

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [7]:
!pip install sentencepiece
import importlib
import transformers
importlib.reload(transformers)

<module 'transformers' from '/usr/local/lib/python3.12/dist-packages/transformers/__init__.py'>

In [8]:
import sentencepiece
print('sentencepiece version:', sentencepiece.__version__)

sentencepiece version: 0.2.2


In [9]:
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed', use_fast=False)

dataset = UrduOCRDataset('data/labels.csv', processor)

sample = dataset[0]
print('Sample pixel_values shape:', sample['pixel_values'].shape)
print('Sample labels shape:', sample['labels'].shape)
print('Dataset is working correctly!')

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, test_size]
)
print(f'Training samples: {train_size}')
print(f'Testing samples: {test_size}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Dataset loaded: 200 samples
Sample pixel_values shape: torch.Size([3, 384, 384])
Sample labels shape: torch.Size([128])
Dataset is working correctly!
Training samples: 160
Testing samples: 40
